# LightGCN

## Load Data

In [ ]:
import gdown, zipfile, os

# Skip if already downloaded
if not os.path.exists('dataset/processed'):
    FILE_ID = '1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6'
    gdown.download(id=FILE_ID, output='processed.zip', quiet=False)
    with zipfile.ZipFile('processed.zip', 'r') as z:
        z.extractall('dataset/')
    os.remove('processed.zip')
    print('Downloaded.')
else:
    print('Already exists, skipping download.')

Already exists, skipping download.


In [ ]:
import pandas as pd
import numpy as np
import pickle
import torch
import torch.nn as nn
import scipy.sparse as sp

train_df = pd.read_csv('dataset/processed/train.csv')
val_df   = pd.read_csv('dataset/processed/val.csv')
test_df  = pd.read_csv('dataset/processed/test.csv')
games_df = pd.read_csv('dataset/processed/games_cleaned.csv')

with open('dataset/processed/id_maps.pkl', 'rb') as f:
    maps = pickle.load(f)

n_users = len(maps['user2idx'])
n_items = len(maps['item2idx'])
print(f'n_users: {n_users:,} | n_items: {n_items:,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

n_users: 272,184 | n_items: 21,922
Train: 18,151,978 | Val: 272,184 | Test: 272,184


## Build the Interaction Graph

LightGCN treats users and items as nodes in a bipartite graph, users to every item they have rated by edges. We represent this as a symmetric normalised adjacency matrix so that nodes with many connections don't dominate aggregation.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

def build_norm_adj(train_df, n_users, n_items):
    """
    Build the symmetric normalised adjacency matrix used in LightGCN
    """
    N = n_users + n_items
    rows_ui = train_df['user_idx'].values
    cols_ui = train_df['item_idx'].values + n_users  # shift item ids past user ids

    # both directions: user->item and item->user
    rows = np.concatenate([rows_ui, cols_ui])
    cols = np.concatenate([cols_ui, rows_ui])
    data = np.ones(len(rows), dtype=np.float32)

    A = sp.csr_matrix((data, (rows, cols)), shape=(N, N))

    # degree-normalise: D^{-1/2} A D^{-1/2}
    degrees    = np.array(A.sum(axis=1)).flatten()
    d_inv_sqrt = np.where(degrees > 0, 1.0 / np.sqrt(degrees), 0.0)
    D_inv_sqrt = sp.diags(d_inv_sqrt)
    A_norm     = D_inv_sqrt @ A @ D_inv_sqrt

    # convert to a PyTorch sparse tensor for GPU matrix multiplication
    A_coo   = A_norm.tocoo()
    indices = torch.from_numpy(np.stack([A_coo.row, A_coo.col])).long()
    values  = torch.from_numpy(A_coo.data).float()
    return torch.sparse_coo_tensor(indices, values, (N, N)).to(device)

print('Building normalised adjacency matrix...')
adj = build_norm_adj(train_df, n_users, n_items)
print(f'Adjacency matrix size: {n_users + n_items:,} x {n_users + n_items:,}')

Using device: cuda
Building normalised adjacency matrix...
Adjacency matrix size: 294,106 x 294,106


## LightGCN Model

Graph propagation works by repeatedly averaging a node's embedding with its neighbours' embeddings. Each layer extends the reach one hop further, so layer 1 captures direct neighbours, layer 2 captures neighbours-of-neighbours, and so on. The final embedding is simply the mean of all these layer snapshots, blending local identity with progressively wider neighbourhood context. The only thing actually learned is the starting embedding for each node so there are no weight matrices or activation functions anywhere in the process.

In [ ]:
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64, n_layers=3):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers
        self.embedding = nn.Embedding(n_users + n_items, emb_dim)
        nn.init.xavier_uniform_(self.embedding.weight)

    def propagate(self, adj):
        """Run graph propagation and return final user & item embeddings."""
        embs       = self.embedding.weight   # layer-0: raw embeddings
        all_layers = [embs]

        for _ in range(self.n_layers):
            embs = torch.sparse.mm(adj, embs) # each node averages its neighbours
            all_layers.append(embs)

        # mean-pool across layers — captures multi-hop structure
        final = torch.stack(all_layers, dim=0).mean(dim=0)
        return final[:self.n_users], final[self.n_users:]

    def forward(self, adj, users, pos_items, neg_items):
        """BPR loss for a batch of (user, positive item, negative item) triples."""
        user_embs, item_embs = self.propagate(adj)

        u  = user_embs[users]
        pi = item_embs[pos_items]
        ni = item_embs[neg_items]

        # score = dot product; BPR wants pos score > neg score
        pos_scores = (u * pi).sum(dim=1)
        neg_scores = (u * ni).sum(dim=1)
        bpr_loss   = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-10).mean()

        # L2 regularisation on the batch embeddings
        reg_loss = (u.norm(2).pow(2) + pi.norm(2).pow(2) + ni.norm(2).pow(2)) / len(users)
        return bpr_loss, reg_loss

    def get_scores(self, adj, user_idx):
        """Predicted score for one user across all items."""
        with torch.no_grad():
            user_embs, item_embs = self.propagate(adj)
            scores = user_embs[user_idx] @ item_embs.T
        return pd.Series(scores.cpu().numpy(), index=np.arange(self.n_items))

## Training

BPR sampling for each step we draw (user, positive item, negative item) triples: the positive is an item the user rated; the negative is a random item they have not seen. The model learns to score positives above negatives.

In [ ]:
# hyperparameters
EMB_DIM         = 64
N_LAYERS        = 3
LR              = 0.001
REG_WEIGHT      = 0.0001
BATCH_SIZE      = 4096
N_EPOCHS        = 10
STEPS_PER_EPOCH = 1000

model     = LightGCN(n_users, n_items, emb_dim=EMB_DIM, n_layers=N_LAYERS).to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=LR)

# pre-build positive item lookup for fast BPR sampling
user_positives = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
user_pos_list  = train_df.groupby('user_idx')['item_idx'].apply(list).to_dict()

rng = np.random.default_rng(42)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Model parameters: 18,822,784


In [ ]:
def sample_batch(batch_size, rng):
    """Sample a batch of (user, pos_item, neg_item) triples for BPR"""
    users     = rng.integers(0, n_users, size=batch_size)
    pos_items = np.array([rng.choice(user_pos_list[u]) for u in users])
    neg_items = np.empty(batch_size, dtype=np.int64)
    for i, u in enumerate(users):
        while True:
            neg = rng.integers(0, n_items)
            if neg not in user_positives[u]:
                neg_items[i] = neg
                break
    return (
        torch.tensor(users,     dtype=torch.long, device=device),
        torch.tensor(pos_items, dtype=torch.long, device=device),
        torch.tensor(neg_items, dtype=torch.long, device=device),
    )

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for _ in range(STEPS_PER_EPOCH):
        users, pos_items, neg_items = sample_batch(BATCH_SIZE, rng)
        bpr_loss, reg_loss = model(adj, users, pos_items, neg_items)
        loss = bpr_loss + REG_WEIGHT * reg_loss
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        epoch_loss += loss.item()
    print(f'Epoch {epoch:2d} | Loss: {epoch_loss / STEPS_PER_EPOCH:.4f}')
print('Training complete!')

Epoch  1 | Loss: 0.6891
Epoch  2 | Loss: 0.6204
Epoch  3 | Loss: 0.5743
Epoch  4 | Loss: 0.5384
Epoch  5 | Loss: 0.5089
Epoch  6 | Loss: 0.4840
Epoch  7 | Loss: 0.4626
Epoch  8 | Loss: 0.4440
Epoch  9 | Loss: 0.4275
Epoch 10 | Loss: 0.4127
Training complete!


## Evaluation

Shared evaluation protocol used by all models. For each user, ranks K candidate items (1 positive test item + 99 random negatives) and computes HR@10, NDCG@10, MRR.

In [ ]:
def evaluate(model, adj, train_df, test_df, n_negatives=99, K=10, seed=42):
    """
    model: trained LightGCN
    For each test user: rank 1 positive + n_negatives random unseen items.
    """
    model.eval()
    rng = np.random.default_rng(seed)
    all_items = np.arange(n_items)
    seen = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

    hits, ndcgs, mrrs = [], [], []

    for _, row in test_df.iterrows():
        uid      = int(row['user_idx'])
        pos_item = int(row['item_idx'])

        # sample negatives (items user hasn't seen)
        user_seen  = seen.get(uid, set()) | {pos_item}
        candidates = np.setdiff1d(all_items, list(user_seen))

        n_to_sample = min(len(candidates), n_negatives)
        neg_items   = rng.choice(candidates, size=n_to_sample, replace=False)

        score_series  = model.get_scores(adj, uid)
        items_to_rank = np.append(neg_items, pos_item)
        scores        = score_series.reindex(items_to_rank).fillna(0).values
        ranked        = items_to_rank[np.argsort(-scores)]
        pos_rank = np.where(ranked == pos_item)[0][0] + 1  # 1-indexed

        hits.append(1 if pos_rank <= K else 0)
        ndcgs.append(np.log(2) / np.log(pos_rank + 1) if pos_rank <= K else 0)
        mrrs.append(1 / pos_rank if pos_rank <= K else 0)

    return {
        f'HR@{K}':   round(np.mean(hits), 4),
        f'NDCG@{K}': round(np.mean(ndcgs), 4),
        f'MRR@{K}':  round(np.mean(mrrs), 4),
    }

## Results

In [ ]:
print('Evaluating LightGCN...')
results = evaluate(model, adj, train_df, test_df)
print(results)

Evaluating LightGCN...
{'HR@10': 0.2790, 'NDCG@10': 0.1923, 'MRR@10': 0.1671}


In [ ]:
all_results = {
    'Bayesian popularity': {'HR@10': 0.1833, 'NDCG@10': 0.0946, 'MRR@10': 0.0679},
    'SVD':                 {'HR@10': 0.1920, 'NDCG@10': 0.1034, 'MRR@10': 0.0766},
    'Item-CF':             {'HR@10': 0.2446, 'NDCG@10': 0.1749, 'MRR@10': 0.1524},
    'LightGCN':            results,
}
results_df = pd.DataFrame(all_results).T
print(results_df.to_string())

                       HR@10  NDCG@10  MRR@10
Bayesian popularity   0.1833   0.0946  0.0679
SVD                   0.1920   0.1034  0.0766
Item-CF               0.2446   0.1749  0.1524
LightGCN              0.2790   0.1923  0.1671


## Sample Recommendations


In [ ]:
# Add game names for readability
idx_to_bggid = maps['idx2item']
games_lookup = games_df.set_index('BGGId')[['Name', 'AvgRating']]
seen = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

def get_recommendations(user_idx, top_k=10):
    score_series = model.get_scores(adj, user_idx)
    score_series[list(seen.get(user_idx, set()))] = -np.inf
    top_items = score_series.nlargest(top_k).index.tolist()
    bgg_ids = [idx_to_bggid[i] for i in top_items]
    recs = games_lookup.loc[bgg_ids].reset_index()
    recs['LightGCN'] = score_series[top_items].values
    return recs

# Sample 3 users
sample_users = test_df['user_idx'].sample(3, random_state=42).tolist()
for uid in sample_users:
    username  = maps['idx2user'][uid]
    num_rated = len(train_df[train_df['user_idx'] == uid])
    print(f'\nTop 10 recommendations for user {username} | Rated {num_rated} games:')
    print(get_recommendations(uid).to_string(index=False))


Top 10 recommendations for user VincentParadise | Rated 7 games:
 BGGId                                          Name  AvgRating  LightGCN
167791                          Terraforming Mars      8.41879    9.2341
224517                         Brass: Birmingham      8.66562    9.1872
182028  Through the Ages: A New Story of Civilization  8.38861    9.0543
169786                                    Scythe      8.21978    8.9910
174430                             Gloomhaven          8.79441    8.9102
120677                           Terra Mystica          8.13399    8.8734
266192                               Wingspan          8.10855    8.8201
 31260                               Agricola          7.92801    8.7655
 84876                   The Castles of Burgundy      8.12711    8.7102
124361                               Concordia          8.12090    8.6894

Top 10 recommendations for user Anrab | Rated 7 games:
 BGGId                                          Name  AvgRating  LightGCN
1